# 数学与优化：从形状到梯度，再到收敛

目标：检查矩阵形状、手推梯度与数值梯度是否一致，并直接看到过大学习率造成发散。所有数据为人工构造；使用 CPU，不需要模型或网络。

先读[数学与统计](../01-concepts/math-and-statistics/README.md)，完整函数在[foundations_core.py](../05-code/foundations_core.py)。按顺序运行全部单元。

In [1]:
from pathlib import Path
import sys, platform
import numpy as np
here = Path.cwd().resolve()
repo = next(p for p in [here, *here.parents] if (p / '10-Knowledge').is_dir())
sys.path.insert(0, str(repo / '10-Knowledge' / '01-ai-foundations' / '05-code'))
from foundations_core import *
np.set_printoptions(precision=6, suppress=True)
print('Python:', platform.python_version(), 'NumPy:', np.__version__)
print('Data: synthetic teaching examples; no model download or API call')

Python: 3.12.13 NumPy: 2.5.2
Data: synthetic teaching examples; no model download or API call


## 1. 同一个预测问题的输入、标签与形状

用直线 $y=1+2x$ 生成8个点。设计矩阵第一列是1、第二列是x，权重应该恢复到[1,2]。`(n,1)`标签减`(n,)`预测会广播成`(n,n)`，因此函数明确拒绝这种形状。

In [2]:
x = np.linspace(-1, 1, 8)
X = np.column_stack([np.ones_like(x), x])
y = 1 + 2*x
w = np.array([0.2, -0.3])
loss, analytic = mse_gradient(X, y, w)
print('X:', X.shape, 'y:', y.shape, 'w:', w.shape)
print('prediction:', X @ w)
print('loss:', loss, 'gradient:', analytic)
try:
    mse_gradient(X, y[:, None], w)
except ValueError as error:
    print('Shape error correctly rejected:', error)
else:
    raise AssertionError('Wrong shape was accepted')

X: (8, 2) y: (8,) w: (2,)
prediction: [ 0.5       0.414286  0.328571  0.242857  0.157143  0.071429 -0.014286
 -0.1     ]
loss: 1.4535714285714283 gradient: [-0.8      -0.985714]
Shape error correctly rejected: Expected X:(n,d), y:(n,), w:(d,)


## 2. 用两次轻微扰动验证每一维导数

目标是 $L=\|Xw-y\|^2/(2n)$。解析梯度是 $X^T(Xw-y)/n$。每次只改变一个参数，用中心差分检查，而不把差分当训练算法。

In [3]:
eps = 1e-6
numeric = np.zeros_like(w)
for j in range(len(w)):
    direction = np.zeros_like(w); direction[j] = eps
    numeric[j] = (mse_gradient(X,y,w+direction)[0] - mse_gradient(X,y,w-direction)[0])/(2*eps)
print('analytic:', analytic)
print('numeric :', numeric)
print('max difference:', np.max(abs(numeric-analytic)))
assert np.allclose(analytic, numeric, atol=1e-8)

analytic: [-0.8      -0.985714]
numeric : [-0.8      -0.985714]
max difference: 1.8576951088533633e-10


## 3. 梯度下降是否真的恢复参数

我们固定学习率0.2、迭代300次。这里是良态凸二次问题，不能把这次成功推广成任意神经网络都可用同一学习率。

In [4]:
trained = np.zeros(2)
history = []
for step in range(300):
    current_loss, grad = mse_gradient(X, y, trained)
    if step in [0, 9, 49, 299]:
        history.append((step+1, current_loss))
    trained -= .2 * grad
print('selected losses:', history)
print('learned weights:', trained)
assert np.allclose(trained, [1.,2.], atol=1e-8)
for lr in [.2, 2.2]:
    scalar = 1.
    values = []
    for _ in range(8):
        values.append(round(scalar,5))
        scalar -= lr*scalar
    print('L(w)=w^2/2, learning_rate=',lr,'w sequence:',values)

selected losses: [(1, 1.357142857142857), (10, 0.17982299238959312), (50, 0.0001315477016889142), (300, 4.570867168541883e-24)]
learned weights: [1. 2.]
L(w)=w^2/2, learning_rate= 0.2 w sequence: [1.0, 0.8, 0.64, 0.512, 0.4096, 0.32768, 0.26214, 0.20972]
L(w)=w^2/2, learning_rate= 2.2 w sequence: [1.0, -1.2, 1.44, -1.728, 2.0736, -2.48832, 2.98598, -3.58318]


## 4. 概率基率与实验单位

1%故障率、90%检出率、5%误报率时，告警后的故障概率远低于90%。这不是检测器“算错了”，是正向条件概率与后验不同。

In [5]:
prior, sensitivity, false_positive = .01, .9, .05
posterior = prior*sensitivity / (prior*sensitivity + (1-prior)*false_positive)
print('P(fault | alarm) =', round(posterior,6))
assert .15 < posterior < .16

P(fault | alarm) = 0.153846


## 观察与边界

参数恢复到[1,2]、差分误差很小，说明这份线性损失和梯度实现一致。学习率2.2时参数绝对值变大，验证了该一维二次问题的稳定步长条件。数据无噪声、模型形式已知，这并未验证泛化；继续做[泛化与校准实验](02-generalization-and-calibration.ipynb)。

练习：把x扩大100倍，先预测原学习率是否仍稳定，再标准化输入观察变化。不要为了得到好曲线修改测试标签。